<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/58/Uber_logo_2018.svg/1024px-Uber_logo_2018.svg.png" alt="UBER LOGO" width="50%" />

# UBER Pickups 

# import libraries

In [1]:
import pandas as pd
import plotly.express as px
import os

# Read data

In [2]:
location = pd.read_csv('data/taxi-zone-lookup.csv')
print('Lines / Columns : ', location.shape)

Lines / Columns :  (265, 3)


In [3]:
files = ["data/uber-raw-data-apr14.csv", "data/uber-raw-data-may14.csv", "data/uber-raw-data-jun14.csv", 'data/uber-raw-data-jul14.csv', 'data/uber-raw-data-aug14.csv', 'data/uber-raw-data-sep14.csv']

raw_2014 = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print('Lines / Columns : ', raw_2014.shape)

Lines / Columns :  (4534327, 4)


In [4]:
raw_2015 = pd.read_csv('data/uber-raw-data-janjune-15.csv')
print('Lines / Columns : ', raw_2015.shape)

Lines / Columns :  (14270479, 4)


In [5]:
# Lowercase dataframes
dataframes = {'location':location, 'raw_2014':raw_2014, 'raw_2015':raw_2015}
for name, df in dataframes.items():
    df.columns = df.columns.str.lower() # lowercase columns names
    for col in df.select_dtypes(include=["object", 'string']).columns: # avoid warning for multi execution
        df[col] = df[col].str.lower()
        print()
    print('-------------------',name,'-------------------')
    print(df.head(1))



------------------- location -------------------
   locationid borough            zone
0           1     ewr  newark airport


------------------- raw_2014 -------------------
          date/time     lat      lon    base
0  4/1/2014 0:11:00  40.769 -73.9549  b02512



------------------- raw_2015 -------------------
  dispatching_base_num          pickup_date affiliated_base_num  locationid
0               b02617  2015-05-17 09:47:00              b02617         141


In [6]:
# os.makedirs('./c') # create directory then deactivate

In [7]:
location.to_csv("c/location.csv", index=False)

# Merge raw_2015 and location

In [8]:
raw_2015 = raw_2015.merge(location[['locationid', "borough", "zone"]], left_on="locationid", right_on="locationid", how="left")
raw_2015.head(1)

,dispatching_base_num,pickup_date,affiliated_base_num,locationid,borough,zone
0,b02617,2015-05-17 09:47:00,b02617,141,manhattan,lenox hill west


# Treat missing values

La référence du taxi est inutile pour kmeans. On remplace par une étiquette explicite.

In [9]:
raw_2015['affiliated_base_num'] = raw_2015['affiliated_base_num'].fillna('unknown') # drop lines where values are missing. 
missing_values = raw_2015.isnull().sum()/raw_2015.shape[0]*100
print(missing_values)

dispatching_base_num    0.0
pickup_date             0.0
affiliated_base_num     0.0
locationid              0.0
borough                 0.0
zone                    0.0
dtype: float64


# Treat datetime

In [10]:
cols = [raw_2014["date/time"], raw_2015["pickup_date"]]

converted = [pd.to_datetime(c, format=f) for c, f in zip(cols, ("%m/%d/%Y %H:%M:%S", "%Y-%m-%d %H:%M:%S")) # zip to use c and f together
]

raw_2014["date/time"], raw_2015["pickup_date"] = converted
print(raw_2014["date/time"].dtypes) # datetime64[ns]
print(raw_2015["pickup_date"].dtypes)
print([c.dtype for c in converted])  # # M8[ns]

datetime64[ns]
datetime64[ns]
[dtype('<M8[ns]'), dtype('<M8[ns]')]


# Analyze one hour on one day

Les bases b02617 et b02764 se superposent. Let's record bases and search for duplicated values.

In [11]:
pickup_14 = raw_2014
pickup_14.to_csv("c/2014.csv", index=False)

In [12]:
pickup = raw_2015
pickup.to_csv("c/2015.csv", index=False)

In [13]:
import json
import os
import urllib.request

url = "https://data.cityofnewyork.us/api/geospatial/8meu-9t5y?method=export&format=GeoJSON"
urllib.request.urlretrieve(url, "c/taxi_zones.geojson")
print(os.path.abspath("c/taxi_zones.geojson"))

c:\Users\LN\Documents\Jedha\Projets\UBER_PICKUPS\c\taxi_zones.geojson


In [14]:
with open("c/taxi_zones.geojson") as f:
    geojson = json.load(f)
print(geojson["features"][0]["properties"])

{'shape_area': '0.0007823067885', 'locationid': '1', 'shape_leng': '0.116357453189', 'zone': 'Newark Airport', 'borough': 'EWR'}


# Merge pickup et taxi_zones.geojson

In [15]:
import geopandas as gpd

taxi_zones = gpd.read_file("c/taxi_zones.geojson")
centroids = taxi_zones.geometry.to_crs(epsg=2263).centroid.to_crs(epsg=4326) # avoid warning le centroïde est calculé en degrés (CRS géographique), donc légèrement imprécis. Pour l’éviter, il faut reprojeter, calculer, puis revenir en lon/lat :
taxi_zones["lon"] = centroids.x
taxi_zones["lat"] = centroids.y
taxi_zones["locationid"] = taxi_zones["locationid"].astype(int)

In [ ]:
pickup = pickup.merge(taxi_zones[["locationid", "lon", "lat"]], left_on="locationid", right_on="locationid", how="left")
pickup.head(1)

,dispatching_base_num,pickup_date,affiliated_base_num,locationid,borough,zone,lon,lat
0,b02617,2015-05-17 09:47:00,b02617,141,manhattan,lenox hill west,-73.959635,40.766948


# Merge pickup et pickup_14

In [ ]:
pickup = pickup.merge(pickup_14[["dispatching_base_num", "lon", "lat"]], left_on="dispatching_base_num", right_on="base", how="left")
pickup.head(1)

In [17]:
df.to_csv("c/df.csv", index=False)

In [18]:
df['pickup_date'].dtype

dtype('<M8[ns]')

# Search for duplicated values

In [19]:
# duplicated_values = bases[bases.duplicated(subset=["lat", "lon"], keep=False)]
# print(duplicated_values.sort_values(['base', "lat", "lon"]))

In [20]:
# with pd.option_context("display.precision", 15):
#     print(bases_median)

There are no duplicated values, only values very similar. Des points proches signalent justement une forte densité de pickups, donc une hot-zone. Les supprimer ferait perdre cette information et fausserait les clusters (KMeans comme DBSCAN).